# 1. Business Understanding

## 1.1 Contexte métier

Dans le secteur des télécommunications, la fidélisation des clients constitue un enjeu important. Lorsqu'un client résilie son abonnement ou quitte l'opérateur, l'entreprise perd les revenus associés à ce client et doit éventuellement engager de nouvelles dépenses pour acquérir un client de remplacement.

Le **churn**, ou attrition client, désigne le fait qu'un client quitte l'entreprise. L'objectif de ce projet est d'exploiter les données historiques des clients afin d'identifier les clients présentant un risque de départ.

Le cas métier choisi est **Telco Customer Churn**. Il s'agit d'un problème de classification supervisée binaire dont la variable cible est `Churn`.

L'objectif n'est pas uniquement d'obtenir une bonne performance prédictive. Comme les prédictions peuvent être utilisées pour orienter des actions de fidélisation, le modèle doit également être **explicable**. Le décideur doit pouvoir comprendre pourquoi un client est considéré comme présentant un risque de départ.

Le projet suit la méthodologie **CRISP-DM (Cross-Industry Standard Process for Data Mining)** :

1. Business Understanding
2. Data Understanding
3. Data Preparation
4. Modeling
5. Evaluation
6. Deployment

---

## 1.2 Problématique métier

L'entreprise dispose de différentes informations concernant ses clients, notamment leur ancienneté, leur contrat, les services souscrits, leur méthode de paiement et leurs charges.

Cependant, elle ne sait pas directement quels clients sont susceptibles de quitter prochainement l'opérateur.

La problématique métier est donc :

> **Comment identifier les clients présentant un risque élevé de churn suffisamment tôt afin de permettre à l'entreprise de mettre en place des actions de fidélisation ciblées, tout en comprenant les raisons qui conduisent le modèle à attribuer ce niveau de risque ?**

Cette problématique comporte trois dimensions :

* **Prédiction** : identifier les clients susceptibles de partir ;
* **Explication** : comprendre les facteurs qui influencent la prédiction ;
* **Décision** : permettre au décideur de prioriser les actions de fidélisation.

---

## 1.3 Objectifs métier

Le projet poursuit les objectifs métier suivants :

### Objectif 1 — Détecter les clients à risque

Identifier les clients ayant une forte probabilité de quitter l'opérateur.

### Objectif 2 — Réduire le churn

Utiliser les prédictions afin de permettre à l'entreprise d'intervenir avant le départ du client et de mettre en place des actions de fidélisation.

### Objectif 3 — Prioriser les actions

Les ressources de l'entreprise étant limitées, tous les clients ne peuvent pas nécessairement faire l'objet d'une action de fidélisation. Le système doit donc permettre d'identifier les clients prioritaires.

### Objectif 4 — Comprendre les facteurs associés au churn

Identifier les caractéristiques qui contribuent le plus au risque de départ afin de permettre au décideur de mieux comprendre le comportement des clients.

### Objectif 5 — Aider à la prise de décision

Fournir des prédictions accompagnées d'explications compréhensibles par un utilisateur non spécialiste du Machine Learning.

---

## 1.4 Objectifs Data Mining

Le problème métier est formulé comme un problème de **classification supervisée binaire**.

La variable cible `Churn` représente le comportement du client :

* `0` : le client reste chez l'opérateur ;
* `1` : le client quitte l'opérateur.

Le système devra donc apprendre une fonction permettant d'estimer :

$$
P(Churn=1 \mid X)
$$

où \(X\) représente les caractéristiques du client.

Le résultat attendu pour chaque client est donc une **probabilité de churn**, qui pourra ensuite être transformée en une décision selon un seuil choisi.

Les objectifs Data Mining sont :

* préparer les données automatiquement ;
* entraîner plusieurs modèles de classification ;
* optimiser leurs hyperparamètres ;
* comparer leurs performances ;
* sélectionner un modèle adapté au problème métier ;
* analyser l'effet du seuil de décision ;
* expliquer les prédictions globales et individuelles ;
* analyser l'influence des variables sensibles ;
* fournir des résultats exploitables par un décideur.

Les modèles étudiés seront :

* Régression Logistique ;
* Arbre de Décision ;
* K plus proches voisins (KNN) ;
* Random Forest ;
* XGBoost.

---

## 1.5 Parties prenantes et utilisation des résultats

Les principaux utilisateurs des résultats peuvent être :

| Partie prenante       | Besoin                                                              |
| --------------------- | ------------------------------------------------------------------- |
| Responsable marketing | Identifier les clients à cibler pour les campagnes de fidélisation  |
| Service client        | Identifier les clients nécessitant une intervention                 |
| Direction             | Suivre le churn et évaluer l'efficacité des actions de fidélisation |
| Data Scientist        | Évaluer, interpréter et améliorer les modèles                       |
| Responsable IT/Data   | Intégrer et maintenir la solution                                   |

L'application web devra donc présenter les résultats sous une forme compréhensible et interactive.

Pour un client donné, le décideur devra pouvoir consulter :

* sa probabilité de churn ;
* la décision obtenue selon le seuil choisi ;
* les variables ayant le plus contribué à la prédiction ;
* une explication SHAP : permet d'expliquer précisément la contribution de chaque variable à la prédiction d'un client.
* une explication LIME : construit une approximation locale du modèle autour du client étudié afin d'identifier les variables qui expliquent sa décision.
* une synthèse textuelle de la prédiction.

---

## 1.6 Coût des erreurs de classification

Les erreurs de classification n'ont pas le même impact métier.

### Faux négatif

Un **faux négatif (FN)** correspond à un client qui va réellement quitter l'entreprise mais que le modèle classe comme client ne présentant pas de risque.

$$
Réel = 1,\quad Prédit = 0
$$

Cette erreur peut entraîner la perte d'un client qui aurait éventuellement pu être fidélisé.

Dans le cadre de ce projet, nous utiliserons l'hypothèse suivante :

$$
C_{FN}=200
$$

### Faux positif

Un **faux positif (FP)** correspond à un client qui restera dans l'entreprise mais que le modèle considère comme étant à risque.

$$
Réel = 0,\quad Prédit = 1
$$

L'entreprise peut alors engager inutilement des ressources pour tenter de fidéliser ce client.

Nous utiliserons l'hypothèse :

$$
C_{FP}=20
$$

Ces valeurs sont des **hypothèses de travail** destinées à permettre l'analyse du coût des erreurs. Elles ne constituent pas des coûts réels fournis par l'entreprise.

Le coût total pourra être estimé par :

$$
Coût_{total} = FN \times 200 + FP \times 20
$$

Puisque :

$$
C_{FN} > C_{FP}
$$

la détection des clients qui vont réellement partir constitue une priorité. Le **rappel (Recall)** sera donc particulièrement important dans l'évaluation du modèle.

---

## 1.7 Critères de succès

La réussite du projet sera évaluée selon plusieurs dimensions.

### Performance prédictive

Les objectifs techniques minimaux sont :

* **AUC-ROC ≥ 0,80** ;
* **Recall ≥ 70 %** sur la classe `Churn`.

D'autres métriques seront également utilisées :

* précision ;
* F1-score ;
* PR-AUC lorsque le déséquilibre des classes le justifie.

### Coût métier

Le modèle doit permettre de limiter les erreurs coûteuses, en particulier les faux négatifs.

Le seuil de décision devra donc être étudié afin d'identifier le compromis entre :

* rappel ;
* précision ;
* nombre de faux positifs ;
* nombre de faux négatifs ;
* coût total estimé.

### Explicabilité

La solution doit permettre d'expliquer les décisions du modèle :

* globalement, pour comprendre le comportement général du modèle ;
* individuellement, pour comprendre la prédiction d'un client donné.

Les méthodes **SHAP** et **LIME** seront utilisées pour cette analyse.

### Équité

Le modèle devra faire l'objet d'une analyse critique concernant l'utilisation éventuelle de variables sensibles, notamment :

* `gender` ;
* `SeniorCitizen`.

L'objectif est de vérifier si ces variables jouent un rôle important dans les prédictions et d'identifier d'éventuelles différences de comportement du modèle entre les groupes.

### Utilité métier

Le résultat final doit être compréhensible par un décideur non spécialiste et permettre de tirer des actions concrètes des explications produites.

---

## 1.8 Contraintes du projet

Le projet doit respecter plusieurs contraintes.

### Contraintes liées aux données

* Le jeu de données doit être téléchargé automatiquement ;
* les données peuvent contenir des valeurs manquantes ;
* les variables sont numériques et catégorielles ;
* les classes peuvent être déséquilibrées ;
* les variables doivent être correctement préparées avant l'entraînement.

### Contraintes liées à la modélisation

Plusieurs modèles doivent être entraînés et comparés :

* Régression Logistique ;
* Arbre de Décision ;
* KNN ;
* Random Forest ;
* XGBoost.

Les hyperparamètres doivent également pouvoir être optimisés.

### Contraintes liées à l'explicabilité

L'application doit fournir des explications globales et locales à l'aide de SHAP et LIME.

### Contraintes liées à l'application

La solution finale doit être accessible à travers une interface web interactive permettant notamment de visualiser les données, les performances des modèles, les courbes ROC, les matrices de confusion, les effets du seuil et les explications.

### Contraintes éthiques

Les variables potentiellement sensibles doivent être analysées afin d'identifier leur influence éventuelle sur les décisions du modèle.

---

## 1.9 Hypothèses métier

Avant l'analyse des données, plusieurs hypothèses peuvent être formulées. Elles seront vérifiées dans les phases suivantes de CRISP-DM.

### H1 — Type de contrat

Les clients ayant un contrat `Month-to-month` pourraient présenter un risque de churn plus élevé que ceux ayant un contrat de plus longue durée.

### H2 — Ancienneté

Les clients ayant une faible ancienneté (`tenure`) pourraient être davantage susceptibles de quitter l'opérateur.

### H3 — Services

Le type de service Internet et les services souscrits pourraient être associés au risque de churn.

### H4 — Paiement

La méthode de paiement pourrait présenter une relation avec le comportement de churn.

### H5 — Charges

Le montant des charges mensuelles (`MonthlyCharges`) et le montant total payé (`TotalCharges`) pourraient être associés au risque de départ.

Ces hypothèses ne sont pas considérées comme des conclusions. Elles devront être vérifiées à partir des données et des résultats des modèles.

---

## 1.10 Décisions attendues

L'objectif final n'est pas uniquement de produire une prédiction, mais de transformer cette prédiction en information utile pour le décideur.

Pour chaque client, le système devra fournir une probabilité de churn.

Par exemple :

$$
P(Churn=1)=0.82
$$

signifie que le modèle estime une probabilité élevée de départ.

Cette information pourra être utilisée pour :

1. identifier les clients prioritaires ;
2. déclencher une action de fidélisation ;
3. sélectionner les clients à contacter ;
4. comprendre les facteurs associés au risque ;
5. adapter les actions de fidélisation au profil du client.

Le seuil de décision ne sera pas nécessairement fixé à `0.5`. Il devra être analysé en fonction du compromis entre le rappel, la précision et le coût des erreurs.

La logique décisionnelle peut donc être représentée par :

$$
\text{Données client}
\rightarrow
\text{Modèle}
\rightarrow
\text{Probabilité de churn}
\rightarrow
\text{Seuil}
\rightarrow
\text{Risque}
\rightarrow
\text{Action}
$$

---

## 1.11 Compromis performance — interprétabilité

Un objectif important du projet consiste à comparer la performance prédictive et l'interprétabilité.

Un modèle complexe tel que **XGBoost** peut fournir de très bonnes performances mais être plus difficile à interpréter directement.

À l'inverse, une **Régression Logistique** est généralement plus simple à interpréter mais peut offrir des performances différentes.

Le choix final ne devra donc pas être basé uniquement sur le modèle ayant la meilleure performance.

Le décideur devra considérer le compromis entre :

* performance prédictive ;
* rappel ;
* coût des erreurs ;
* explicabilité ;
* équité ;
* facilité d'utilisation.

Les méthodes SHAP et LIME permettront notamment d'améliorer l'interprétation des modèles plus complexes.

---

## 1.12 Résultat attendu de la phase Business Understanding

À l'issue de cette phase, le problème peut être résumé ainsi :

> **Construire une solution de classification binaire capable d'identifier les clients susceptibles de quitter un opérateur télécom, tout en fournissant des explications globales et individuelles permettant au décideur de comprendre les prédictions et de prendre des actions de fidélisation adaptées.**

Le succès de la solution sera évalué non seulement par ses performances prédictives, mais également par son coût métier, son explicabilité, son équité et son utilité pour la prise de décision.

Cette définition servira de référence pour les cinq phases suivantes de la méthodologie CRISP-DM.


## 2.1 Chargement des données

Le jeu de données utilisé dans ce projet est le **Telco Customer Churn**.

Il contient des informations relatives aux clients d'un opérateur télécom, notamment leurs caractéristiques démographiques, les services souscrits, leur ancienneté et les informations liées à leur facturation.

La variable cible `Churn` indique si le client a quitté l'entreprise :

- `Yes` : le client a quitté l'entreprise ;
- `No` : le client est toujours client.

Afin de rendre l'application reproductible, le jeu de données est téléchargé automatiquement à partir de la plateforme OpenML.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_openml

# Pour assurer la reproductibilité
RANDOM_STATE = 42

# Chargement automatique du dataset Telco Customer Churn
raw = fetch_openml(data_id=42178, as_frame=True)

df = raw.frame.copy()

print("Dataset chargé avec succès.")
print("Dimensions :", df.shape)

ModuleNotFoundError: No module named 'pandas'

## 2.2 Description des données

Cette étape permet d'obtenir une première vision globale du jeu de données. Nous analysons sa dimension, les types des variables ainsi que leurs principales statistiques descriptives.

In [ ]:
print("Dimensions du dataset :", df.shape)

display(df.head())

In [ ]:
df.info()

In [ ]:
print("Variables disponibles :")

for i, col in enumerate(df.columns, 1):
    print(f"{i}. {col}")

In [ ]:
df.describe()

In [ ]:
df.describe(include="all").T

## 2.3 Analyse de la qualité des données

Avant toute préparation, la qualité du jeu de données est vérifiée. Nous recherchons notamment les doublons, les valeurs vides ou mal représentées, les types incohérents et les catégories présentes dans les variables qualitatives.

In [ ]:
nb_duplicates = df.duplicated().sum()

print("Nombre de lignes dupliquées :", nb_duplicates)

In [ ]:
df[df.duplicated()].head()

In [ ]:
# Recherche des chaînes vides ou composées uniquement d'espaces
empty_strings = {}

for col in df.select_dtypes(include="object").columns:
    count = df[col].astype(str).str.strip().eq("").sum()
    if count > 0:
        empty_strings[col] = count

print("Valeurs vides détectées :")
print(empty_strings)

In [ ]:
print(
    "TotalCharges avec espaces :",
    df["TotalCharges"].astype(str).str.strip().eq("").sum()
)

## 2.4 Analyse des valeurs manquantes

La présence de valeurs manquantes peut affecter la qualité des modèles de classification. Nous identifions donc le nombre et le pourcentage de valeurs manquantes pour chaque variable.

Les traitements des valeurs manquantes seront réalisés dans la phase **Data Preparation**, afin de ne pas modifier prématurément les données originales.

In [ ]:
missing_count = df.isnull().sum()

missing_count = missing_count[missing_count > 0].sort_values(ascending=False)

print("Valeurs manquantes :")
display(missing_count)

In [ ]:
missing_percent = (df.isnull().mean() * 100)

missing_table = pd.DataFrame({
    "Nombre de valeurs manquantes": df.isnull().sum(),
    "Pourcentage (%)": missing_percent
})

missing_table = missing_table[
    missing_table["Nombre de valeurs manquantes"] > 0
].sort_values(
    "Pourcentage (%)",
    ascending=False
)

display(missing_table)

## 2.5 Analyse du déséquilibre des classes

La variable cible du problème est `Churn`. Nous étudions la répartition des clients ayant quitté l'entreprise (`Yes`) et de ceux qui sont restés (`No`).

In [ ]:
print("Répartition de la variable cible :")

target_counts = df["Churn"].value_counts()

display(target_counts)

In [ ]:
target_percent = df["Churn"].value_counts(normalize=True) * 100

target_table = pd.DataFrame({
    "Effectif": target_counts,
    "Pourcentage (%)": target_percent.round(2)
})

display(target_table)

In [ ]:
plt.figure(figsize=(7, 5))

sns.countplot(data=df, x="Churn")

plt.title("Répartition des classes de Churn")
plt.xlabel("Churn")
plt.ylabel("Nombre de clients")

plt.show()

In [ ]:
churn_rate = (
    df["Churn"].value_counts(normalize=True)
    .get("Yes", 0) * 100
)

print(f"Taux de churn : {churn_rate:.2f}%")
print(f"Taux de non-churn : {100 - churn_rate:.2f}%")

In [ ]:
if churn_rate < 40:
    print(
        "Conclusion : la classe Churn présente une représentation minoritaire. "
        "Le dataset est déséquilibré et cette caractéristique devra être prise "
        "en compte lors de la modélisation et de l'évaluation."
    )
else:
    print(
        "Conclusion : les classes sont relativement équilibrées."
    )

2.6 Analyse exploratoire des données — EDA

In [ ]:
# Copie de travail pour l'analyse
df_eda = df.copy()

# Conversion de TotalCharges uniquement pour l'analyse
df_eda["TotalCharges"] = pd.to_numeric(
    df_eda["TotalCharges"],
    errors="coerce"
)

# Identification des variables numériques
numeric_cols = df_eda.select_dtypes(
    include=["int64", "float64", "int32", "float32"]
).columns.tolist()

# Identification des variables catégorielles
categorical_cols = df_eda.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("Variables numériques :")
print(numeric_cols)

print("\nVariables catégorielles :")
print(categorical_cols)

In [ ]:
# 2.6.2 Distribution des variables numériques
df_eda[numeric_cols].describe().T

In [ ]:
for col in numeric_cols:

    plt.figure(figsize=(8, 4))

    sns.histplot(
        data=df_eda,
        x=col,
        kde=True
    )

    plt.title(f"Distribution de {col}")
    plt.xlabel(col)
    plt.ylabel("Fréquence")

    plt.tight_layout()
    plt.show()

In [ ]:
# 2.6.3 Distribution des variables numériques selon le churn

plt.figure(figsize=(8, 5))

sns.histplot(
    data=df_eda,
    x="tenure",
    hue="Churn",
    kde=True,
    element="step"
)

plt.title("Distribution de l'ancienneté selon le Churn")
plt.xlabel("Ancienneté (mois)")
plt.ylabel("Nombre de clients")

plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

sns.boxplot(
    data=df_eda,
    x="Churn",
    y="MonthlyCharges"
)

plt.title("MonthlyCharges selon le Churn")
plt.xlabel("Churn")
plt.ylabel("Frais mensuels")

plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

sns.boxplot(
    data=df_eda,
    x="Churn",
    y="TotalCharges"
)

plt.title("TotalCharges selon le Churn")
plt.xlabel("Churn")
plt.ylabel("Total des frais")

plt.show()

2.6.4 Analyse des variables catégorielles

In [ ]:
def churn_rate_by_category(df, column):

    result = (
        df.groupby(column)["Churn"]
        .apply(lambda x: (x == "Yes").mean() * 100)
        .sort_values(ascending=False)
    )

    return result

In [ ]:
# Calculer le taux de churn pour chaque type de contrat
contract_churn = (
    df_eda.groupby("Contract")["Churn"]
    .apply(lambda x: (x == "Yes").mean() * 100)
    .sort_values(ascending=False)
)

display(contract_churn)

In [ ]:
plt.figure(figsize=(9, 5))

sns.barplot(
    x=contract_churn.index,
    y=contract_churn.values
)

plt.title("Taux de churn selon le type de contrat")
plt.xlabel("Type de contrat")
plt.ylabel("Taux de churn (%)")

plt.xticks(rotation=20)
plt.show()

In [ ]:
important_categorical = [
    "Contract",
    "InternetService",
    "PaymentMethod",
    "OnlineSecurity",
    "TechSupport",
    "PaperlessBilling",
    "Partner",
    "Dependents"
]
for col in important_categorical:

    rates = churn_rate_by_category(df_eda, col)

    plt.figure(figsize=(9, 5))

    sns.barplot(
        x=rates.index,
        y=rates.values
    )

    plt.title(f"Taux de churn selon {col}")
    plt.xlabel(col)
    plt.ylabel("Taux de churn (%)")

    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()

2.6.7 Corrélations entre variables numériques

In [ ]:
# Matrice de corrélation
correlation_matrix = df_eda[numeric_cols].corr()

display(correlation_matrix)

In [ ]:
# Heatmap
plt.figure(figsize=(8, 6))

sns.heatmap(
    correlation_matrix,
    annot=True,
    cmap="coolwarm",
    fmt=".2f"
)

plt.title("Matrice de corrélation des variables numériques")

plt.tight_layout()
plt.show()

In [ ]:
# Synthèse automatique de l'EDA
eda_summary = []

for col in important_categorical:

    rates = churn_rate_by_category(df_eda, col)

    eda_summary.append({
        "Variable": col,
        "Taux min (%)": rates.min(),
        "Taux max (%)": rates.max(),
        "Écart (%)": rates.max() - rates.min()
    })

eda_summary = pd.DataFrame(eda_summary)

eda_summary = eda_summary.sort_values(
    "Écart (%)",
    ascending=False
)

display(eda_summary)

### 2.7 Synthèse de la Data Understanding

La phase de compréhension des données a permis d'analyser le jeu de données Telco Customer Churn, composé de 7 043 observations et de 20 variables. L'audit a identifié 22 lignes dupliquées. Aucune valeur manquante n'a été détectée directement avec `isnull()`, mais les conversions numériques devront être vérifiées lors de la préparation.

L'analyse de la variable cible montre que 26,54 % des clients ont quitté l'entreprise, contre 73,46 % qui sont restés. Ce déséquilibre justifie l'utilisation de plusieurs métriques d'évaluation, notamment le rappel, le F1-score, l'AUC-ROC et la PR-AUC.

L'analyse exploratoire révèle des différences importantes de taux de churn selon le type de contrat : 42,71 % pour les contrats mensuels, 11,27 % pour les contrats d'un an et 2,83 % pour les contrats de deux ans. Une corrélation positive importante a également été observée entre l'ancienneté (`tenure`) et le montant total facturé (`TotalCharges`).


**3. Data Preparation — Préparation des données:**

In [ ]:
# Créer une copie de travail pour conserver les données originales
df_prep = df.copy()

# Convertir TotalCharges en numérique
# Les valeurs impossibles à convertir deviennent NaN
df_prep["TotalCharges"] = pd.to_numeric(
    df_prep["TotalCharges"],
    errors="coerce"
)

print("Dimensions initiales :", df_prep.shape)
print("Valeurs manquantes après conversion :")
display(df_prep.isnull().sum().sort_values(ascending=False).head(10))

print("Nombre de doublons exacts :", df_prep.duplicated().sum())

In [ ]:
# Variable cible : 1 = client parti, 0 = client resté
y = (df_prep["Churn"].astype(str).str.strip() == "Yes").astype(int)

# Variables explicatives
X = df_prep.drop(columns=["Churn", "customerID"], errors="ignore")

print("Dimensions de X :", X.shape)
print("Dimensions de y :", y.shape)

print("\nRépartition de la cible :")
display(y.value_counts().rename(index={0: "No", 1: "Yes"}))

print("\nTypes des variables explicatives :")
display(X.dtypes)

In [ ]:
# .3 Séparer les données en entraînement et test
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Taille de X_train :", X_train.shape)
print("Taille de X_test  :", X_test.shape)

print("\nTaux de churn dans les données complètes :", round(y.mean() * 100, 2), "%")
print("Taux de churn dans train :", round(y_train.mean() * 100, 2), "%")
print("Taux de churn dans test  :", round(y_test.mean() * 100, 2), "%")

In [ ]:
# 3.4 Imputation, encodage et normalisation

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Identifier les colonnes à partir des données d'entraînement
num_cols = X_train.select_dtypes(include=np.number).columns.tolist()
cat_cols = X_train.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

print("Variables numériques :", num_cols)
print("Variables catégorielles :", cat_cols)

# Pipeline numérique : imputation par médiane + normalisation
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Pipeline catégoriel : imputation par modalité la plus fréquente + encodage
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combiner les deux traitements
preprocess = ColumnTransformer(transformers=[
    ("num", numeric_pipeline, num_cols),
    ("cat", categorical_pipeline, cat_cols)
])

# Ajuster uniquement sur train, puis transformer train et test
X_train_processed = preprocess.fit_transform(X_train)
X_test_processed = preprocess.transform(X_test)

print("Dimensions après préparation :")
print("X_train_processed :", X_train_processed.shape)
print("X_test_processed  :", X_test_processed.shape)

Médiane : permet d'imputer les valeurs numériques manquantes sans être aussi sensible aux valeurs extrêmes que la moyenne.

Modalité la plus fréquente : permet de compléter les valeurs catégorielles manquantes.

One-Hot Encoding : transforme les catégories en colonnes numériques utilisables par les modèles.

StandardScaler : met les variables numériques à une échelle comparable, ce qui est notamment utile pour KNN et la régression logistique.

Ajustement sur X_train uniquement : empêche d'utiliser des informations statistiques du jeu de test pendant l'apprentissage.

4] MODELING: L'objectif de cette phase est de construire plusieurs modèles de classification capables de prédire le départ des clients. Cinq algorithmes seront comparés : la régression logistique, l'arbre de décision, K-Nearest Neighbors (KNN), Random Forest et XGBoost.

Cinq algorithmes sont étudiés :

Logistic Regression : modèle de classification de référence.
Decision Tree : arbre de décision permettant de représenter des règles de classification.
KNN : classification basée sur la proximité entre les observations.
Random Forest : ensemble de plusieurs arbres de décision.
XGBoost : algorithme de boosting qui construit successivement des arbres pour améliorer les prédictions.

In [ ]:
# Vérification des dimensions
print("Train :", X_train_processed.shape)
print("Test  :", X_test_processed.shape)

# Vérifier les valeurs manquantes après transformation
print("Valeurs manquantes dans train :",
      np.isnan(X_train_processed).sum())

print("Valeurs manquantes dans test :",
      np.isnan(X_test_processed).sum())

# Vérifier la distribution de la cible
print("\nRépartition de y_train :")
print(y_train.value_counts(normalize=True).round(3))

print("\nRépartition de y_test :")
print(y_test.value_counts(normalize=True).round(3))

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import roc_auc_score

print("Les modèles et outils de validation sont prêts.")

In [ ]:
# 4.2 Définition des modèles

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# Calcul du poids de la classe positive (Churn = 1)
nb_no = (y_train == 0).sum()
nb_yes = (y_train == 1).sum()

scale_pos_weight = nb_no / nb_yes

print("Clients sans churn :", nb_no)
print("Clients avec churn :", nb_yes)
print("Ratio des classes :", round(scale_pos_weight, 2))

# Définition des modèles
models = {
    "Logistic Regression": LogisticRegression(
        class_weight="balanced",
        max_iter=2000,
        random_state=42
    ),

    "Decision Tree": DecisionTreeClassifier(
        class_weight="balanced",
        random_state=42
    ),

    "KNN": KNeighborsClassifier(),

    "Random Forest": RandomForestClassifier(
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ),

    "XGBoost": XGBClassifier(
        scale_pos_weight=scale_pos_weight,
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    )
}

print("\nLes cinq modèles sont prêts.")

In [ ]:
# 4.3 Validation croisée stratifiée

from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

print("Validation croisée stratifiée : 5 plis")
print("Critère de sélection : AUC-ROC")

In [ ]:
# 4.4 Recherche des meilleurs hyperparamètres

from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import loguniform

param_distributions = {
    "Logistic Regression": {
        "C": [0.01, 0.1, 1, 10],
        "solver": ["liblinear"]
    },

    "Decision Tree": {
        "max_depth": [3, 5, 8, 12, None],
        "min_samples_split": [2, 10, 20],
        "min_samples_leaf": [1, 5, 10]
    },

    "KNN": {
        "n_neighbors": [5, 7, 9, 11, 15],
        "weights": ["uniform", "distance"],
        "p": [1, 2]
    },

    "Random Forest": {
        "n_estimators": [100, 200, 300],
        "max_depth": [None, 10, 20],
        "min_samples_split": [2, 5, 10],
        "min_samples_leaf": [1, 2, 4],
        "max_features": ["sqrt", "log2"]
    },

    "XGBoost": {
        "n_estimators": [100, 200, 300],
        "max_depth": [3, 4, 6],
        "learning_rate": [0.01, 0.05, 0.1],
        "subsample": [0.8, 1.0],
        "colsample_bytree": [0.8, 1.0]
    }
}

print("Les espaces de recherche sont définis.")

In [ ]:
# Recherche et comparaison des modèles

import pandas as pd
import time

best_models = {}
results = []

for name, model in models.items():
    print(f"\nOptimisation de : {name}")
    start = time.time()

    search = RandomizedSearchCV(
        estimator=model,
        param_distributions=param_distributions[name],
        n_iter=5,
        scoring="roc_auc",
        cv=cv,
        random_state=42,
        n_jobs=-1,
        refit=True,
        verbose=1
    )

    search.fit(X_train_processed, y_train)

    best_models[name] = search.best_estimator_

    results.append({
        "Modèle": name,
        "Meilleure AUC-ROC CV": search.best_score_,
        "Meilleurs paramètres": search.best_params_,
        "Temps (secondes)": round(time.time() - start, 2)
    })

    print("Meilleure AUC-ROC :", round(search.best_score_, 4))
    print("Meilleurs paramètres :", search.best_params_)

results_df = pd.DataFrame(results)
results_df = results_df.sort_values(
    by="Meilleure AUC-ROC CV",
    ascending=False
).reset_index(drop=True)

print("\nClassement des modèles :")
display(results_df)

In [ ]:
# 4.5 Sélection du meilleur modèle selon l'AUC-ROC CV

best_model_name = results_df.loc[0, "Modèle"]
best_model = best_models[best_model_name]

print("Meilleur modèle :", best_model_name)
print(
    "AUC-ROC moyenne en validation croisée :",
    round(results_df.loc[0, "Meilleure AUC-ROC CV"], 4)
)

### 4.6. Bilan de la modélisation

Les cinq algorithmes de classification ont été entraînés et optimisés à l'aide de `RandomizedSearchCV`, avec une validation croisée stratifiée à cinq plis. L'AUC-ROC a été utilisée comme critère principal de sélection des hyperparamètres.

Les résultats montrent que **XGBoost obtient la meilleure AUC-ROC moyenne, avec un score de 0,8494**, suivi de Logistic Regression (0,8462) et de Random Forest (0,8439).

XGBoost est donc retenu comme meilleur candidat initial. Cependant, la sélection définitive du modèle sera effectuée après l'évaluation sur les données de test, en prenant en compte plusieurs métriques, notamment la précision, le rappel, le F1-score et l'AUC-ROC, ainsi que les exigences métier liées à la détection des clients susceptibles de résilier.


# 5. Evaluation — Évaluation des modèles

Cette phase vise à évaluer les performances des modèles optimisés sur les données de test, qui n'ont pas été utilisées pendant l'entraînement ni pendant la recherche des hyperparamètres.

Les métriques étudiées sont l'AUC-ROC, la PR-AUC, la précision, le rappel, le F1-score et le Gini. Les résultats permettront de comparer les modèles et d'étudier leur capacité à détecter les clients susceptibles de résilier leur abonnement.

Les analyses suivantes porteront sur les courbes ROC et Precision-Recall, les matrices de confusion, le choix du seuil de décision, le coût métier, l'explicabilité avec SHAP et LIME, ainsi que l'équité des prédictions (*fairness*).


In [ ]:
# 5.1 Évaluation initiale des cinq modèles sur le jeu de test

import pandas as pd
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score
)

evaluation_results = []

for name, model in best_models.items():

    # Probabilité que le client résilie (classe 1)
    y_proba = model.predict_proba(X_test_processed)[:, 1]

    # Prédiction avec le seuil standard de 0.5
    y_pred = (y_proba >= 0.5).astype(int)

    auc = roc_auc_score(y_test, y_proba)
    pr_auc = average_precision_score(y_test, y_proba)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    accuracy = accuracy_score(y_test, y_pred)

    evaluation_results.append({
        "Modèle": name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1-score": f1,
        "AUC-ROC": auc,
        "PR-AUC (Average Precision)": pr_auc,
        "Gini": 2 * auc - 1
    })

evaluation_df = pd.DataFrame(evaluation_results)

evaluation_df = evaluation_df.sort_values(
    by="AUC-ROC",
    ascending=False
).reset_index(drop=True)

# Afficher les scores avec 4 décimales
display(evaluation_df.round(4))

### 5.2. Analyse des courbes ROC

Les courbes ROC permettent de comparer la capacité des modèles à distinguer les clients qui résilient leur abonnement de ceux qui restent. Elles représentent le taux de vrais positifs en fonction du taux de faux positifs pour différents seuils de classification.

L'AUC-ROC résume la performance de chaque courbe : une valeur plus élevée indique généralement une meilleure capacité de discrimination.


In [ ]:
# 5.2 Tracer les courbes ROC des cinq modèles

import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, roc_auc_score

plt.figure(figsize=(9, 6))

for name, model in best_models.items():
    y_proba = model.predict_proba(X_test_processed)[:, 1]

    fpr, tpr, thresholds = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)

    plt.plot(fpr, tpr, label=f"{name} (AUC = {auc:.3f})")

plt.plot([0, 1], [0, 1], linestyle="--", label="Classification aléatoire")

plt.xlabel("Taux de faux positifs (FPR)")
plt.ylabel("Taux de vrais positifs (TPR)")
plt.title("Comparaison des courbes ROC")
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# 5.3 Matrices de confusion

import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

for i, (name, model) in enumerate(best_models.items()):
    y_proba = model.predict_proba(X_test_processed)[:, 1]
    y_pred = (y_proba >= 0.5).astype(int)

    cm = confusion_matrix(y_test, y_pred, labels=[0, 1])

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=["Pas de churn", "Churn"]
    )

    disp.plot(ax=axes[i], colorbar=False, values_format="d")
    axes[i].set_title(name)

# Supprimer le sixième emplacement inutilisé
fig.delaxes(axes[5])

plt.tight_layout()
plt.show()

In [ ]:
# 5.4 Étude des seuils pour XGBoost

import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score

xgb_model = best_models["XGBoost"]
y_proba_xgb = xgb_model.predict_proba(X_test_processed)[:, 1]

threshold_results = []

for threshold in [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60]:
    y_pred_threshold = (y_proba_xgb >= threshold).astype(int)

    threshold_results.append({
        "Seuil": threshold,
        "Precision": precision_score(
            y_test, y_pred_threshold, zero_division=0
        ),
        "Recall": recall_score(
            y_test, y_pred_threshold, zero_division=0
        ),
        "F1-score": f1_score(
            y_test, y_pred_threshold, zero_division=0
        )
    })

threshold_df = pd.DataFrame(threshold_results)
display(threshold_df.round(4))

In [ ]:
# 5.5 Calcul du coût métier selon le seuil

import pandas as pd
from sklearn.metrics import confusion_matrix

COST_FN = 200
COST_FP = 20

cost_results = []

for threshold in threshold_df["Seuil"]:
    y_pred = (y_proba_xgb >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_test,
        y_pred,
        labels=[0, 1]
    ).ravel()

    total_cost = fn * COST_FN + fp * COST_FP

    cost_results.append({
        "Seuil": threshold,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp,
        "Coût total": total_cost
    })

cost_df = pd.DataFrame(cost_results)

display(cost_df)

# Seuil qui minimise le coût estimé
best_cost_row = cost_df.loc[cost_df["Coût total"].idxmin()]

print("\nSeuil minimisant le coût :", best_cost_row["Seuil"])
print("Coût total estimé :", best_cost_row["Coût total"])

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 5))

plt.plot(
    cost_df["Seuil"],
    cost_df["Coût total"],
    marker="o"
)

plt.xlabel("Seuil de décision")
plt.ylabel("Coût métier estimé")
plt.title("Évolution du coût métier selon le seuil")
plt.grid(alpha=0.3)
plt.show()

### 5.5. Optimisation du seuil selon le coût métier

Une analyse du coût métier a été réalisée pour différents seuils de décision du modèle XGBoost. Le coût total a été calculé en attribuant un coût de 200 unités à chaque faux négatif et de 20 unités à chaque faux positif.

Parmi les seuils testés, le seuil de **0,20** présente le coût total estimé le plus faible, soit **14 300 unités**. À ce seuil, le modèle détecte 363 clients susceptibles de résilier sur les 374 résiliations réelles du jeu de test, tout en générant 605 fausses alertes.

Ce résultat illustre le compromis entre la détection des résiliations et le coût des actions de fidélisation inutiles. Le seuil retenu dépend des coûts métier et devra être réévalué si ces hypothèses changent.


In [ ]:
# Vérification du coût métier au seuil de 0.20

row = cost_df.loc[cost_df["Seuil"] == 0.20].iloc[0]

print("TN :", int(row["TN"]))
print("FP :", int(row["FP"]))
print("FN :", int(row["FN"]))
print("TP :", int(row["TP"]))

calculated_cost = (
    COST_FN * row["FN"]
    + COST_FP * row["FP"]
)

print("Coût recalculé :", calculated_cost)
print("Coût du tableau :", row["Coût total"])

### 5.6. Analyse des courbes Precision-Recall

Les courbes Precision-Recall représentent la précision en fonction du rappel pour différents seuils de décision. Elles sont particulièrement utiles pour évaluer les modèles lorsque les classes sont déséquilibrées, comme dans notre problème de prédiction du churn.

La comparaison des courbes permettra d'étudier le compromis entre la détection des clients qui résilient leur abonnement et la réduction des fausses alertes.


In [ ]:
# Courbes Precision-Recall des cinq modèles

import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, average_precision_score

plt.figure(figsize=(9, 6))

for name, model in best_models.items():
    y_proba = model.predict_proba(X_test_processed)[:, 1]

    precision, recall, thresholds = precision_recall_curve(
        y_test, y_proba
    )

    ap = average_precision_score(y_test, y_proba)

    plt.plot(recall, precision, label=f"{name} (AP = {ap:.3f})")

plt.xlabel("Recall (Rappel)")
plt.ylabel("Precision (Précision)")
plt.title("Comparaison des courbes Precision-Recall")
plt.legend(loc="best")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# Curseur interactif du seuil pour XGBoost

import ipywidgets as widgets
from IPython.display import display, clear_output
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

# Probabilités déjà calculées sur le jeu de test
xgb_model = best_models["XGBoost"]
y_proba_xgb = xgb_model.predict_proba(X_test_processed)[:, 1]

threshold_slider = widgets.FloatSlider(
    value=0.50,
    min=0.10,
    max=0.90,
    step=0.01,
    description="Seuil :",
    continuous_update=False,
    readout_format=".2f",
    layout=widgets.Layout(width="500px")
)

output = widgets.Output()

def update_threshold(change=None):
    threshold = threshold_slider.value
    y_pred = (y_proba_xgb >= threshold).astype(int)

    precision = precision_score(
        y_test, y_pred, zero_division=0
    )
    recall = recall_score(
        y_test, y_pred, zero_division=0
    )
    f1 = f1_score(
        y_test, y_pred, zero_division=0
    )

    with output:
        clear_output(wait=True)

        print(f"Seuil de décision : {threshold:.2f}")
        print(f"Précision : {precision:.4f}")
        print(f"Rappel : {recall:.4f}")
        print(f"F1-score : {f1:.4f}")

        cm = confusion_matrix(y_test, y_pred, labels=[0, 1])

        disp = ConfusionMatrixDisplay(
            confusion_matrix=cm,
            display_labels=["Pas de churn", "Churn"]
        )

        disp.plot(values_format="d", colorbar=False)
        plt.title(f"Matrice de confusion — Seuil {threshold:.2f}")
        plt.show()

threshold_slider.observe(update_threshold, names="value")

display(threshold_slider, output)
update_threshold()

### 5.7 Explainable AI (Module 5)

In [ ]:
import sys; sys.path.insert(0, "../src")
from churn import explain as E
import shap, matplotlib.pyplot as plt
names = E.get_feature_names(preprocess)
Xte = E.to_dense(X_test_processed)
perm = E.permutation_table(best_models, preprocess, X_test, y_test)
E.plot_permutation(perm, "perm.png"); 
_, sv = E.compute_shap(best_models["XGBoost"], Xte, names)
shap.plots.beeswarm(sv, max_display=15)

ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
top = np.abs(sv.values).mean(0).argsort()[::-1][:4]
for j in top:
    shap.plots.scatter(sv[:, int(j)], color=sv)

In [ ]:
bg = E.to_dense(preprocess.transform(X_train.sample(1000, random_state=42)))
r = E.explain_one(best_models["XGBoost"], preprocess, X_test.iloc[[0]], bg, 0.20)
print(r["proba"], r["decision"]); print(r["sentence"])
shap.plots.waterfall(r["shap"], max_display=12)
r["lime"].as_pyplot_figure(label=1); display(r["comparison"])

In [ ]:
display(E.fairness_table(best_models["XGBoost"], preprocess, X_test, y_test, 0.20))
display(E.counterfactual_shift(best_models["XGBoost"], preprocess, X_test))

# 6) Déploiement

Le code de déploiement a été extrait du notebook vers des modules Python versionnés :

| Étape du notebook | Fichier du projet |
|---|---|
| 6.1 Sauvegarde du modèle, du prétraitement, des métadonnées | `src/churn/train.py` |
| Schéma de l'interface (`ui_schema.json`) | `src/churn/features.py` |
| API FastAPI (`main.py`) | `api/main.py` |
| Interface Streamlit (`app_streamlit.py`) | `app/streamlit_app.py` |

```bash
python -m churn.train                      # entraîne et sauvegarde les artefacts
uvicorn api.main:app --reload              # API sur http://127.0.0.1:8000
streamlit run app/streamlit_app.py         # interface sur http://localhost:8501
```

Voir le `README.md` pour le détail.